Here we check the number of duplicates.

I found in encounter_id 0 duplicates, so each row contains its own encounter with a person at a hospital for some treatment.
In patient_nbr I found 71518 unique values, so there are 71518 unique persons in the database.
For the rest the duplicates seemed fine

In [1]:
import numpy, pandas, matplotlib, sklearn, seaborn, ipykernel

import pandas as pd

df = pd.read_csv("diabetic_data.csv", na_values="?")

# For each column: how many rows repeat a value that already appeared earlier in that column
dup_counts = df.apply(lambda col: col.duplicated().sum())

# Missing values per column (count and percentage)
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)

# Put it all in one table
summary = pd.DataFrame({
    "n_unique": df.nunique(),
    "n_duplicates": dup_counts,
    "n_missing": missing_counts,
    "pct_missing": missing_pct
}).sort_values("n_missing", ascending=False)

print(summary.to_string())

print(df.shape)

/var/folders/t2/z2d0j1ns56l8gkc11rwm53pw0000gn/T/ipykernel_11321/4032011713.py:5: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("diabetic_data.csv", na_values="?")


                          n_unique  n_duplicates  n_missing  pct_missing
weight                           9        101756      98569        96.86
max_glu_serum                    3        101762      96420        94.75
A1Cresult                        3        101762      84748        83.28
medical_specialty               72        101693      49949        49.08
payer_code                      17        101748      40256        39.56
race                             5        101760       2273         2.23
diag_3                         789        100976       1423         1.40
diag_2                         748        101017        358         0.35
diag_1                         716        101049         21         0.02
encounter_id                101766             0          0         0.00
troglitazone                     2        101764          0         0.00
tolbutamide                      2        101764          0         0.00
pioglitazone                     4        101762   

Here we check the different possible values per column in this dataset.

In the race column we see 2273 NaN values and 1506 Other.

Gender contains 3 Unknown/Invalid values, which are not noted as missing values in python here.

We see that there are 161 individuals in the [0-10) age group which may be due to birth, we should maybe filter for these people as well. Or also separate in a different dataset.

Weight has 98569  values, which is 96,8% of the dataset

admission_type_id has no missing values or weird values

discharge_disposition_id no missing /weird values

admission_source_id no missing values

time_in_hospital no missing values

Payer_code has 40256 NaN values, which is 39,5% of the database.

medical_specialty has 49949 NaN values which is 49,1% of the dataset. However the names should be checked, which are not good and which should be combined with an umbrella specialty like the two separate ones 'Orthopedics' and 'Orthopedics-Reconstructive'. Domain knowledge should be applied here.

num_lab_procedures no missing values.

num_procedures no missing values. It contained 46652 values with 0.

num_medications no missing values

number_outpatient no missing values

number_emergency no missing values

number_inpatient no missing values.

diag_1 has 21 missing values (0.02%)

diag_2 has 358 missing values (0.35%)

diag_3 has 1423 missing values (1.40%)

number_diagnoses no missing values

max_glu_serum has 96420 (94.75%) missing values

A1Cresult has 84748 (83.28%) missing values.

metformin, repaglinide, nateglinide, chlorpropamide, glimepiride, acetohexamide, glipizide, glyburide, tolbutamide, pioglitazone, rosiglitazone, acarbose, miglitol, troglitazone, tolazamide no missing values.

examide, citoglipton no missing values, but only contains no.

insulin, glyburide-metformin, glipizide-metformin  no missing values.

glimepiride-pioglitazone, metformin-rosiglitazone, metformin-pioglitazone no missing values, only contains no values

change no missing values.

diabetesMed no missing values.

readmitted no missing values.


In [11]:
df = pd.read_csv("diabetic_data.csv", na_values="?")

pd.set_option("display.max_rows", None)   # show all rows, no "..."

cols_with_missing = df.columns

for col in cols_with_missing:
    print(f"\n=== {col} ({df[col].nunique()} unique values) ===")
    print(df[col].value_counts(dropna=False))

cols = ["diag_1", "diag_2", "diag_3"]
missing = df[cols].isnull().sum()
percent = (missing / len(df) * 100).round(2)

print(pd.DataFrame({"missing": missing, "percent": percent}))

print(df.shape)

/var/folders/t2/z2d0j1ns56l8gkc11rwm53pw0000gn/T/ipykernel_15233/2776093993.py:1: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("diabetic_data.csv", na_values="?")



=== encounter_id (101766 unique values) ===
encounter_id
2278392      1
149190       1
64410        1
500364       1
16680        1
35754        1
55842        1
63768        1
12522        1
15738        1
28236        1
36900        1
40926        1
42570        1
62256        1
73578        1
77076        1
84222        1
89682        1
148530       1
150006       1
150048       1
182796       1
183930       1
216156       1
221634       1
236316       1
248916       1
250872       1
252822       1
253380       1
253722       1
260166       1
293058       1
293118       1
325848       1
325866       1
326028       1
358776       1
377268       1
383430       1
419304       1
421194       1
449142       1
450210       1
464994       1
486156       1
498030       1
537834       1
544194       1
550098       1
584136       1
590346       1
591996       1
604188       1
623880       1
630342       1
654918       1
669954       1
676416       1
676422       1
679722       1
682494      

From the results above we find that there are only 3197 observations with weight included

The code below shows that encounter_id and patient_nbr have only numeral values. Furthermore, patient_nbr has 6 values that contain 3 digits and 31 values that contain 4 digits.

In [8]:
for col in ["encounter_id", "patient_nbr"]:
    values = df[col].astype(str).str.strip()
    non_numeric = df[pd.to_numeric(df[col], errors="coerce").isna()]
    
    print(f"{col}: {len(non_numeric)} non-numeric rows")
    if len(non_numeric) > 0:
        print(non_numeric[[col]].head(20))

    lengths = df[col].astype(str).str.len()
    counts = lengths[lengths <= 4].value_counts().sort_index()

    print(f"\n{col}:")
    for length in range(1, 5):
        print(f"  {length} digits: {counts.get(length, 0)}")

encounter_id: 0 non-numeric rows

encounter_id:
  1 digits: 0
  2 digits: 0
  3 digits: 0
  4 digits: 0
patient_nbr: 0 non-numeric rows

patient_nbr:
  1 digits: 0
  2 digits: 0
  3 digits: 6
  4 digits: 31
